# 01 — Exploratory Data Analysis (EDA)

**Goal:** Load UAH-DRIVESET-v1, understand its structure, and visualise raw signals.

Classes: `0=normal`, `1=aggressive`, `2=economic (drowsy)`

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from data_loader import UAHDriveSetLoader, LABEL_MAP, INT_TO_LABEL

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = Path('../data/raw/UAH-DRIVESET-v1')
FIG_DIR  = Path('../results/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Loading

In [ ]:
loader = UAHDriveSetLoader(DATA_DIR, merge=True)
loader.load(verbose=True)
print(f'\nTotal trips: {len(loader)}')

In [ ]:
summary = loader.summary()
print(summary.to_string())

## 2. Trip Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

summary['behavior'].value_counts().plot.bar(ax=axes[0], color=['#4C72B0','#DD8452','#55A868'])
axes[0].set_title('Behavior Distribution'); axes[0].set_xlabel('')

summary['road_type'].value_counts().plot.bar(ax=axes[1], color=['#8172B2','#C44E52'])
axes[1].set_title('Road Type Distribution'); axes[1].set_xlabel('')

summary['driver'].value_counts().sort_index().plot.bar(ax=axes[2], color='steelblue')
axes[2].set_title('Trips per Driver'); axes[2].set_xlabel('')

plt.tight_layout()
fig.savefig(FIG_DIR / 'eda_distribution.png', dpi=150)
plt.show()

## 3. Raw Signal Visualisation

In [ ]:
# One example trip per class
behaviors = ['normal', 'aggressive', 'economic']
colors    = {'normal': '#4C72B0', 'aggressive': '#DD8452', 'economic': '#55A868'}
signal_cols   = ['acc_x_kf', 'acc_y_kf', 'acc_z_kf']
signal_labels = ['Acc X (longitudinal)', 'Acc Y (lateral)', 'Acc Z (vertical)']

fig, axes = plt.subplots(3, 3, figsize=(16, 9), sharex=False)

for col_idx, (sig, sig_lbl) in enumerate(zip(signal_cols, signal_labels)):
    for row_idx, beh in enumerate(behaviors):
        trips = loader.get_by_behavior(beh)
        if not trips:
            continue
        trip = trips[0]
        df = trip.merged if not trip.merged.empty else trip.accel
        ax = axes[col_idx][row_idx]
        t  = df['timestamp'].values
        v  = df[sig].values if sig in df.columns else np.zeros(len(t))
        ax.plot(t, v, color=colors[beh], linewidth=0.6, alpha=0.9)
        ax.set_title(f'{beh.capitalize()} — {sig_lbl}', fontsize=9)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('G')

plt.tight_layout()
fig.savefig(FIG_DIR / 'eda_raw_signals.png', dpi=150)
plt.show()

## 4. Speed Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for beh in behaviors:
    speed_vals = []
    for trip in loader.get_by_behavior(beh):
        df = trip.merged if not trip.merged.empty else trip.gps
        if 'speed_kmh' in df.columns:
            speed_vals.extend(df['speed_kmh'].dropna().values)
    if speed_vals:
        sns.kdeplot(speed_vals, ax=ax, label=beh.capitalize(), color=colors[beh], linewidth=2)

ax.set_xlabel('Speed (km/h)')
ax.set_ylabel('Density')
ax.set_title('Speed Distribution by Behavior')
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / 'eda_speed_distribution.png', dpi=150)
plt.show()

## 5. Acceleration Correlation Heatmap

In [ ]:
# Correlation over the first 5000 rows
combined = []
for trip in loader.trips[:10]:
    df = trip.merged if not trip.merged.empty else trip.accel
    combined.append(df.head(500))

all_df = pd.concat(combined, ignore_index=True)
accel_cols = ['acc_x', 'acc_y', 'acc_z', 'acc_x_kf', 'acc_y_kf', 'acc_z_kf', 'roll', 'pitch', 'yaw']
avail = [c for c in accel_cols if c in all_df.columns]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(all_df[avail].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            square=True, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Sensor Signal Correlation Matrix')
plt.tight_layout()
fig.savefig(FIG_DIR / 'eda_correlation.png', dpi=150)
plt.show()